# Train CatBoost classifier

Input file (place in the same folder):
- `ML_train.xlsx`

Outputs:
- `retrained_catboost_model.cbm`
- `retrained_catboost_model.meta.json`

The model used in the thesis is `final_catboost_model.cbm` (with `final_catboost_model.meta.json`),
provided in this folder. The output names are different so that re-running does not overwrite it.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier, Pool
import optuna
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import json

features = [
    'FBO type',
    'Product type',
    'Operation type',
    'Former Inspection result',
    'FBO Size',
    'Inspection interval (days)'
]
cat_features = [
    'FBO type',
    'Product type',
    'Operation type',
    'Former Inspection result',
    'FBO Size'
]

## 1. Load data

In [ ]:
df = pd.read_excel('ML_train.xlsx', converters={'Product type': str})

# Behaviour change (-3 to 3) grouped into three classes as target variable:
# 0 = improvement, 1 = no change, 2 = deterioration
# (translated back to -1/0/+1 for the ABM via the meta file)

y = df['Behavior Change'].map({-3: 0, -2: 0, -1: 0, 0: 1, 1: 2, 2: 2, 3: 2})

X = df[features].copy()

for c in cat_features:
    X[c] = X[c].astype(str)

print(y.value_counts().sort_index())

## 2. Split dataset

In [ ]:
# 60% training, 20% validation, 20% test

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_val, y_val, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features)

## 3. Class weights

In [ ]:
# Inverse-frequency weights
classes = sorted(y.unique())
train_counts = y_train.value_counts()
total = len(y_train)
class_weights = []
for c in classes:
    class_weights.append(total / (len(classes) * train_counts[c]))
print("class weights:", class_weights)

## 4. Hyperparameter tuning

In [ ]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_seed': 42,
        'loss_function': 'MultiClass',
        'eval_metric': 'MultiClass',
        'class_weights': class_weights,
        'early_stopping_rounds': 50,
        'verbose': False,
    }
    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
  
    preds = model.predict(val_pool).astype(int).ravel()
    return f1_score(y_val, preds, average='macro')
    
 # maximise macro-F1
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print("Best params:", study.best_trial.params)

## 5. Final model 

In [ ]:
best_params = study.best_trial.params
best_params.update({
    'loss_function': 'MultiClass',
    'class_weights': class_weights,
    'random_seed': 42,
    'early_stopping_rounds': 50,
    'verbose': 100,
})
final_model = CatBoostClassifier(**best_params)
final_model.fit(train_pool, eval_set=val_pool, use_best_model=True)

# Performance
val_preds = final_model.predict(val_pool).astype(int).ravel()
print("Validation report:\n", classification_report(y_val, val_preds, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_val, val_preds))

test_preds = final_model.predict(test_pool).astype(int).ravel()
print("Test report:\n", classification_report(y_test, test_preds, zero_division=0))
print("Test confusion matrix:\n", confusion_matrix(y_test, test_preds))
print("Test macro-F1:", f1_score(y_test, test_preds, average='macro'))
print("Test accuracy:", accuracy_score(y_test, test_preds))

## 6. Save the classifier for use in ABM

In [2]:
# final_catboost_model.cbm provided in the folder was trained with this notebook.
# To avoid overwriting it when re-running, the output name is changed as follows:
final_model.save_model('retrained_catboost_model.cbm')

# The ABM reads -1 as improvement, 0 as no change and +1 as deterioration,
# so the model classes (0/1/2) are mapped to these values in the meta file
meta = {
    'feature_order': features,
    'categorical_features': cat_features,
    'target_mapping': {0: -1, 1: 0, 2: 1},
}
with open('retrained_catboost_model.meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

Loading: ML_train.xlsx


[I 2026-09-29 19:26:09,222] A new study created in memory with name: no-name-7771664b-2ddc-4dee-ac97-cd6630cd6ab3


Product type categories: {'0.0312': 2862, '0.0182': 1059, '0.047': 243, '0.0392': 15}
Target counts: {0: 232, 1: 3766, 2: 181}
Class weights: [np.float64(6.011990407673861), np.float64(0.36992769662092373), np.float64(7.666666666666667)]


[I 2026-09-29 19:26:11,678] Trial 0 finished with value: 0.6323087052282217 and parameters: {'iterations': 330, 'depth': 7, 'learning_rate': 0.26857602583020007, 'l2_leaf_reg': 6.37894056509131, 'border_count': 172}. Best is trial 0 with value: 0.6323087052282217.
[I 2026-09-29 19:26:14,518] Trial 1 finished with value: 0.6250241711436759 and parameters: {'iterations': 229, 'depth': 8, 'learning_rate': 0.09543265683719429, 'l2_leaf_reg': 1.573262050439041, 'border_count': 215}. Best is trial 0 with value: 0.6323087052282217.
[I 2026-09-29 19:26:15,685] Trial 2 finished with value: 0.633040086147988 and parameters: {'iterations': 217, 'depth': 4, 'learning_rate': 0.2931327252050374, 'l2_leaf_reg': 5.358104088044586, 'border_count': 142}. Best is trial 2 with value: 0.633040086147988.
[I 2026-09-29 19:26:17,533] Trial 3 finished with value: 0.611187593728698 and parameters: {'iterations': 139, 'depth': 5, 'learning_rate': 0.1588473607163373, 'l2_leaf_reg': 1.8571421547590639, 'border_cou

Best params: {'iterations': 288, 'depth': 8, 'learning_rate': 0.2783585741021404, 'l2_leaf_reg': 5.628912330545075, 'border_count': 87}
0:	learn: 0.8848663	test: 0.8734134	best: 0.8734134 (0)	total: 10.5ms	remaining: 3.02s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.4825127553
bestIteration = 25

Shrink model to first 26 iterations.
Validation report:
               precision    recall  f1-score   support

           0       0.92      1.00      0.96        46
           1       0.97      0.72      0.83       754
           2       0.08      0.53      0.15        36

    accuracy                           0.73       836
   macro avg       0.66      0.75      0.64       836
weighted avg       0.93      0.73      0.80       836

Confusion matrix:
 [[ 46   0   0]
 [  4 543 207]
 [  0  17  19]]
Test report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98        47
           1       0.96      0.70      0.81       75